In [6]:
from pathlib import Path
import numpy as np
import pandas as pd

import os

import json

import reservoirpy as rpy
from reservoirpy.nodes import Reservoir, ScikitLearnNode


atlas_name = "schaefer200"

# dataset preparation

In [7]:
data_path = Path("../data/emofilm_fmri/derivatives/preprocessing/")
parcellated_timeseries_dir = Path(f"../data/emofilm_fmri/derivatives/parcellated_timeseries/{atlas_name}")

ts_files = os.listdir(parcellated_timeseries_dir)
ts_files = [f for f in ts_files if f.endswith(".tsv")]
print(ts_files)
working_dir = Path("../data/emofilm/scratch/")
movie_name = "ToClaireFromSonny"


if not os.path.isfile(working_dir / f"behav_data_{movie_name}.csv"):

    X = []
    subjects = []
    for file in ts_files:
        ts_path = parcellated_timeseries_dir / file
        ts_data = np.loadtxt(ts_path, delimiter=" ") #yes, it's a tsv file with space delimiter. good work on my end
        X.append(ts_data)
        print(f"File: {file}, shape: {ts_data.shape}") #n_timepoints x n_parcels
        subjects.append(file.split("_")[0].split("-")[1]) #assuming filename format is sub-XX_ses-3_task-ToClaireFromSonny_parcellated_timeseries.tsv

    behav_data_path = Path("../data/emofilm_annotation/derivatives/")
    os.listdir(behav_data_path)


    with open(behav_data_path / f"Annot_{movie_name}_stim.json", "r") as f:
        metadata = json.load(f)

    behav_data = pd.read_csv(behav_data_path / f"Annot_{movie_name}_stim.tsv.gz",
        compression="gzip",
        sep="\t",
        header=None
        )

    behav_data.columns = metadata["Columns"]

    behav_data.head()

    os.makedirs(working_dir, exist_ok=True)
    np.savez(working_dir / f"parcellated_timeseries_{atlas_name}", *X)

    behav_data.to_csv(working_dir / f"behav_data_{movie_name}.csv", index=False)
    # movie_start_times_s = []
    # for subject in subjects:
    #     subject = str(subject).zfill(2)
    #     file = f"../data/emofilm_fmri/sub-{subject}/ses-3/func/sub-{subject}_ses-3_task-ToClaireFromSonny_physio.json"
    #     with open(file, "r") as f:
    #         physio_metadata = json.load(f)
    #     movie_start_time_s = physio_metadata["StartTime"]
    #     movie_start_times_s.append(movie_start_time_s)

else:
    X = np.load(working_dir / f"parcellated_timeseries_{atlas_name}.npz")
    X = [X[f'arr_{i}'] for i in range(len(X.files))]
    behav_data = pd.read_csv(working_dir / f"behav_data_{movie_name}.csv")



['sub-S08_ses-3_task-ToClaireFromSonny_space-MNI_desc-ppres_bold_parcellated.tsv', 'sub-S23_ses-3_task-ToClaireFromSonny_space-MNI_desc-ppres_bold_parcellated.tsv', 'sub-S25_ses-3_task-ToClaireFromSonny_space-MNI_desc-ppres_bold_parcellated.tsv', 'sub-S01_ses-3_task-ToClaireFromSonny_space-MNI_desc-ppres_bold_parcellated.tsv', 'sub-S13_ses-3_task-ToClaireFromSonny_space-MNI_desc-ppres_bold_parcellated.tsv', 'sub-S19_ses-3_task-ToClaireFromSonny_space-MNI_desc-ppres_bold_parcellated.tsv']


In [8]:
behav_data.head() # continuous emotional assessments accross time

,Standards,PleasantSelf,SocialNorms,PleasantOther,GoalsOther,Controlled,Predictable,Suddenly,Agent,Urgency,...,Disgust,Happiness,Fear,Regard,Anxiety,Satisfaction,Pride,Surprise,Love,Sad
0,0.085194,0.453663,0.181572,0.424665,0.50657,0.035689,0.025618,-0.079119,-0.445523,-0.200073,...,0.162723,0.471479,-0.313899,-0.142059,-0.855552,0.246556,0.281798,-0.295054,-0.302899,-0.322154
1,0.085194,0.453663,0.276600,0.424665,0.50657,0.035689,0.025618,-0.079119,-0.445523,-0.248635,...,0.162723,0.471479,-0.313899,-0.142059,-0.855552,0.246556,0.271094,-0.295054,-0.302899,-0.322154
2,0.085194,0.453663,0.274289,-0.222434,0.50657,0.035689,-0.211909,-0.079119,-0.445523,-0.433115,...,0.162723,0.471479,-0.313899,-0.142059,-0.855552,0.246556,0.275806,-0.295054,-0.302899,-0.322154
3,0.085194,0.453663,0.296932,0.424665,0.50657,0.035689,-0.447696,-0.079119,-0.445523,-0.671932,...,0.162723,0.471479,-0.313899,-0.105246,-0.855552,0.195112,0.363367,-0.295054,-0.302899,-0.322154
4,0.198084,0.453869,0.347350,0.441406,0.50657,0.035689,-0.476414,-0.079119,-0.445523,-0.723889,...,-0.069440,0.471479,-0.313899,-0.017509,-0.855552,0.184870,0.381034,-0.295054,-0.302899,-0.322154


In [9]:
# Create the reservoir.
reservoir = Reservoir(
    units=300,
    sr=0.9,
    lr=0.3,
    input_scaling=0.5,
)
